# Week 003 — Riutilizzare lo stato

Periodo: **2026-08-17 — 2026-08-21**

Quattro esercizi progressivi su stringhe, aggregazione annidata, sliding window e query online.

## Regole

- Leggi tutti gli esercizi prima di scegliere l'ordine.
- Avvia un timer separato e registra tempi, blocchi e decisioni in `notes.md`.
- Se superi il timebox, annota il punto raggiunto e passa oltre.
- Non usare AI, soluzioni online o autocomplete generativo.
- Modifica soltanto le celle delle funzioni, non i test.
- Esegui i test visibili; Codex valutera' edge case e prestazioni durante la review.


## Mappa della quest

| # | Esercizio | Difficolta' | Timebox | Skill principali |
|---|---|---:|---:|---|
| 1 | Nomi canonici delle feature | Facile | 15 min | String manipulation, set, ordine stabile |
| 2 | Report delle valutazioni modello | Media | 25 min | Aggregazione annidata, predicati esatti, tie-breaking |
| 3 | Sequenza di richieste affidabile | Media | 30 min | Sliding window, conteggio sotto vincolo |
| 4 | Monitor delle code dei worker | Media-difficile | 35 min | Frequency map, query online, transizioni di soglia |

Tempo target complessivo: **105 minuti**.


# Esercizio 1 — Nomi canonici delle feature

**Difficolta':** facile  
**Timebox:** 15 minuti  
**Skill:** string manipulation, deduplicazione, ordine stabile

Implementa `canonical_feature_names(names)`.

`names` e' una lista di stringhe provenienti da sorgenti dati diverse. Restituisci la lista dei nomi canonici unici, nell'ordine della loro prima apparizione valida.

Per ottenere il nome canonico di una stringa:

- rimuovi gli spazi all'inizio e alla fine;
- sostituisci ogni sequenza interna di uno o piu' caratteri di whitespace con un singolo underscore;
- converti tutte le lettere in minuscolo.

Regole:

- ignora le stringhe che risultano vuote dopo la normalizzazione;
- considera duplicati i nomi con la stessa forma canonica;
- non modificare l'input;
- la somma delle lunghezze delle stringhe puo' arrivare a 500.000 caratteri.

### Esempio

```python
canonical_feature_names([" User  Age ", "user\tage", "TOTAL SPEND", "   "])
# Output atteso: ["user_age", "total_spend"]
```


In [45]:
def canonical_feature_names(names):
    """Restituisce i nomi canonici unici in ordine stabile."""
    risultato=[]
    for name in names:
        name=name.strip().lower()
        words=name.split()
        name_clean="_".join(word.strip() for word in words)
        if name_clean =="":
            continue
        else:
            if name_clean not in risultato:
                risultato.append(name_clean)
    return risultato


In [46]:
# Test visibili: non modificarli.
names_test = [" User  Age ", "user\tage", "TOTAL SPEND", "total   spend", "   ", "Country"]
names_snapshot = names_test.copy()
assert canonical_feature_names(names_test) == ["user_age", "total_spend", "country"]
assert canonical_feature_names([]) == []
assert canonical_feature_names([" A\nB ", "a b", "C"]) == ["a_b", "c"]
assert canonical_feature_names(["", " \t "]) == []
assert names_test == names_snapshot
print("Esercizio 1: test visibili superati")


Esercizio 1: test visibili superati


# Esercizio 2 — Report delle valutazioni modello

**Difficolta':** media  
**Timebox:** 25 minuti  
**Skill:** aggregazione annidata, predicati esatti, tie-breaking stabile

Implementa `summarize_model_evaluations(evaluations)`.

Ogni valutazione puo' contenere `team`, `model`, `passed` e `latency_ms`. Restituisci un dizionario indicizzato per team. Ogni team deve avere questa struttura:

```python
{
    "models": {
        model: {
            "runs": numero di valutazioni valide,
            "passed_runs": numero di valutazioni con passed is True,
            "total_latency": somma delle latenze,
        }
    },
    "most_tested_model": modello con piu' valutazioni valide,
}
```

Regole:

- una valutazione e' valida soltanto se `team` e `model` esistono e non valgono `None`;
- `passed_runs` aumenta soltanto quando `passed is True`;
- se `latency_ms` manca o vale `None`, considerala zero;
- in caso di parita' per `most_tested_model`, scegli il modello apparso per primo in una valutazione valida di quel team;
- preserva l'ordine di prima apparizione valida dei team e dei relativi modelli;
- non modificare gli input;
- l'input puo' contenere fino a 100.000 valutazioni.

### Esempio e output atteso

```python
summarize_model_evaluations([
    {"team": "risk", "model": "m1", "passed": True, "latency_ms": 10},
    {"team": "risk", "model": "m2", "passed": 1, "latency_ms": 7},
    {"team": "risk", "model": "m1", "passed": False},
])
# {
#   "risk": {
#     "models": {
#       "m1": {"runs": 2, "passed_runs": 1, "total_latency": 10},
#       "m2": {"runs": 1, "passed_runs": 0, "total_latency": 7},
#     },
#     "most_tested_model": "m1",
#   }
# }
```


In [120]:
def summarize_model_evaluations(evaluations):
    """Aggrega le valutazioni per team e modello."""
    lookup={}
    for evaluation in evaluations:
        team = evaluation.get("team")
        model=evaluation.get("model")
        if team is not None and model is not None:
            passed = 1 if evaluation["passed"] is True else 0
            latency_ms=evaluation.get("latency_ms") if evaluation.get("latency_ms") != None else 0
            if team not in lookup:
                lookup[team]={}
                lookup[team]["models"]={}
                lookup[team]["models"][model]={"runs": 0, "passed_runs": 0, "total_latency": 0}
            if model not in lookup[team]["models"]:
                lookup[team]["models"][model]={"runs": 0, "passed_runs": 0, "total_latency": 0}
            lookup[team]["models"][model]["runs"]+=1
            lookup[team]["models"][model]["passed_runs"]+=passed
            lookup[team]["models"][model]["total_latency"]+=latency_ms
    
    for team, models in lookup.items():
        max_test=0
        best_model=""
        for statistics in models.values():
            for model, info in statistics.items():
                valid_test=info["runs"]
                if valid_test>max_test:
                   max_test=valid_test 
                   best_model=model 
        lookup[team]["most_tested_model"]=best_model

    return lookup
    


In [121]:
# Test visibili: non modificarli.
evaluations_test = [
    {"team": "risk", "model": "m1", "passed": True, "latency_ms": 10},
    {"team": "risk", "model": "m2", "passed": 1, "latency_ms": 7},
    {"team": "risk", "model": "m1", "passed": False},
    {"team": "search", "model": "s1", "passed": True, "latency_ms": None},
    {"team": "search", "model": "s2", "passed": True, "latency_ms": 4},
    {"team": None, "model": "ignored", "passed": True, "latency_ms": 99},
]
evaluations_snapshot = [record.copy() for record in evaluations_test]
assert summarize_model_evaluations(evaluations_test) == {
    "risk": {
        "models": {
            "m1": {"runs": 2, "passed_runs": 1, "total_latency": 10},
            "m2": {"runs": 1, "passed_runs": 0, "total_latency": 7},
        },
        "most_tested_model": "m1",
    },
    "search": {
        "models": {
            "s1": {"runs": 1, "passed_runs": 1, "total_latency": 0},
            "s2": {"runs": 1, "passed_runs": 1, "total_latency": 4},
        },
        "most_tested_model": "s1",
    },
}
assert summarize_model_evaluations([]) == {}
assert summarize_model_evaluations([{"team": "t", "model": None}]) == {}
assert evaluations_test == evaluations_snapshot
print("Esercizio 2: test visibili superati")


Esercizio 2: test visibili superati


# Esercizio 3 — Sequenza di richieste affidabile

**Difficolta':** media  
**Timebox:** 30 minuti  
**Skill:** sliding window, segmenti contigui, complessita'

Implementa `longest_reliable_request_streak(status_codes, max_server_errors)`.

`status_codes` e' una lista di codici HTTP interi. Un codice e' un errore server se e' compreso tra 500 e 599 inclusi. Restituisci la lunghezza massima di un segmento contiguo che contiene al massimo `max_server_errors` errori server.

Regole:

- se `max_server_errors < 0`, solleva `ValueError`;
- una lista vuota produce zero;
- codici fuori dall'intervallo 500-599 non consumano il limite;
- non modificare l'input;
- la funzione deve gestire fino a 200.000 codici in tempo O(n) e spazio ausiliario O(1).

### Esempio

```python
longest_reliable_request_streak([200, 503, 200, 500, 502, 200], 1)
# Output atteso: 3
```


In [17]:
def longest_reliable_request_streak(status_codes, max_server_errors):
    """Restituisce il segmento contiguo affidabile piu' lungo."""
    if max_server_errors < 0:
        raise  ValueError("Max Server Error non può essere negativo")
    left=0
    current_count_error=0
    best=0

    for right, status in enumerate(status_codes):
        if 500 <= status < 600:
            current_count_error+=1
        while current_count_error > max_server_errors:
            if 500<= status_codes[left]< 600:
                current_count_error-=1
            left +=1
        best = max(best, right-left+1)
    return best


In [18]:
# Test visibili: non modificarli.
status_test = [200, 503, 200, 500, 502, 200]
status_snapshot = status_test.copy()
assert longest_reliable_request_streak(status_test, 1) == 3
assert longest_reliable_request_streak([], 2) == 0
assert longest_reliable_request_streak([200, 201, 204], 0) == 3
assert longest_reliable_request_streak([500, 200, 501, 200, 200], 1) == 4
assert longest_reliable_request_streak([500, 501], 0) == 0
assert longest_reliable_request_streak([499, 500, 599, 600], 1) == 2
assert status_test == status_snapshot

try:
    longest_reliable_request_streak([200], -1)
except ValueError:
    pass
else:
    raise AssertionError("max_server_errors < 0 deve sollevare ValueError")

print("Esercizio 3: test visibili superati")


Esercizio 3: test visibili superati


# Esercizio 4 — Monitor delle code dei worker

**Difficolta':** media-difficile  
**Timebox:** 35 minuti  
**Skill:** frequency map, query online, transizioni di soglia

Implementa `process_worker_queues(initial_jobs, overload_limit, operations)`.

`initial_jobs` contiene il nome del worker assegnato a ciascun job inizialmente in coda; lo stesso nome puo' comparire piu' volte. Un worker e' sovraccarico quando il suo numero di job e' strettamente maggiore di `overload_limit`.

Ogni operazione ha uno dei formati seguenti:

- `["enqueue", worker]`: aggiunge un job alla coda del worker;
- `["complete", worker]`: completa un job, se il worker ne ha almeno uno;
- `["load", worker]`: aggiunge al risultato il numero corrente di job del worker;
- `["overloaded"]`: aggiunge al risultato il numero corrente di worker sovraccarichi.

Regole:

- se `overload_limit < 0`, solleva `ValueError`;
- `complete` su un worker con coda vuota o assente non ha effetto;
- i conteggi non possono diventare negativi;
- soltanto `load` e `overloaded` producono elementi nella lista di output;
- preserva l'ordine delle risposte e non modificare gli input;
- input iniziale e operazioni possono contenere complessivamente fino a 200.000 elementi;
- il tempo complessivo atteso e' O(n + q) in media, dove `n` e' la lunghezza di `initial_jobs` e `q` il numero di operazioni.

### Esempio

```python
process_worker_queues(
    ["a", "a", "b"],
    1,
    [
        ["load", "a"],
        ["overloaded"],
        ["complete", "a"],
        ["overloaded"],
        ["enqueue", "b"],
        ["load", "b"],
        ["overloaded"],
    ],
)
# Output atteso: [2, 1, 0, 2, 1]
```


In [33]:
def process_worker_queues(initial_jobs, overload_limit, operations):
    """Elabora aggiornamenti e query sulle code dei worker."""
    if overload_limit < 0:
        raise ValueError("Overload dev'essere positivo")
    result=[]
    workers_jobs={}
    overloaded_worker=0
    for worker in initial_jobs:
        if worker not in workers_jobs:
            workers_jobs[worker]=0
        workers_jobs[worker]+=1
    
    for jobs  in workers_jobs.values():
        if jobs > overload_limit:
            overloaded_worker+=1

    for operation in operations:
        if operation[0] == "enqueue":
            if operation[1] in workers_jobs:
                workers_jobs[operation[1]]+=1
                if workers_jobs[operation[1]]> overload_limit:
                    overloaded_worker+=1
        if operation[0] == "complete":
            if operation[1] in workers_jobs:
                if workers_jobs[operation[1]]>0:
                    workers_jobs[operation[1]]-=1
                    if workers_jobs[operation[1]] <= overload_limit:
                        overloaded_worker-=1
        if operation[0] == "load":
            result.append(workers_jobs.get(operation[1], 0))
        if operation[0] == "overloaded":
            result.append(overloaded_worker)
    return result


In [34]:
# Test visibili: non modificarli.
jobs_test = ["a", "a", "b"]
queue_operations_test = [
    ["load", "a"],
    ["overloaded"],
    ["complete", "a"],
    ["overloaded"],
    ["enqueue", "b"],
    ["load", "b"],
    ["overloaded"],
]
jobs_snapshot = jobs_test.copy()
queue_operations_snapshot = [operation.copy() for operation in queue_operations_test]
assert process_worker_queues(jobs_test, 1, queue_operations_test) == [2, 1, 0, 2, 1]
assert process_worker_queues([], 0, [["load", "x"], ["overloaded"]]) == [0, 0]
assert process_worker_queues(["x"], 0, [["overloaded"], ["complete", "x"], ["overloaded"]]) == [1, 0]
assert process_worker_queues(["x", "x"], 2, [["overloaded"], ["enqueue", "x"], ["overloaded"], ["complete", "x"], ["overloaded"]]) == [0, 1, 0]
assert jobs_test == jobs_snapshot
assert queue_operations_test == queue_operations_snapshot

try:
    process_worker_queues([], -1, [])
except ValueError:
    pass
else:
    raise AssertionError("overload_limit < 0 deve sollevare ValueError")

print("Esercizio 4: test visibili superati")


Esercizio 4: test visibili superati


# Chiusura della quest

Prima di fermarti:

- esegui tutte le celle nell'ordine corretto;
- verifica che tutti i test visibili passino;
- completa tempi, primo blocco e decisione in `notes.md`;
- annota il debrief finale;
- non cancellare i tentativi utili: servono per la review.
